# Silver: fact_marketing_booking

Deduplicate bronze marketing bookings to one row per `booking_id` using latest-record logic, cast raw strings to proper typed columns, and validate referential integrity against dimension tables and web event/session sources.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


SOURCE_TABLE = "voyage.bronze.marketing_bookings"
TARGET_TABLE = "voyage.silver.fact_marketing_booking"


# -------------------------------------------------------------------
# Read bronze source
# -------------------------------------------------------------------

bronze_df = spark.table(SOURCE_TABLE)

print("Bronze row count:", bronze_df.count())


# -------------------------------------------------------------------
# Deduplicate: keep latest record per booking_id
# -------------------------------------------------------------------

silver_df = (
    bronze_df
    .withColumn(
        "_rn",
        F.row_number().over(
            Window.partitionBy("booking_id")
            .orderBy(
                F.col("updated_at").desc(),
                F.col("_airbyte_extracted_at").desc(),
            )
        ),
    )
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .select(
        F.col("booking_id"),
        F.col("tenant_id"),
        F.col("property_id"),
        F.col("user_id"),
        F.col("conversion_id"),
        F.col("session_id"),
        F.col("booking_status"),
        F.to_date(F.col("booking_date")).alias("booking_date"),
        F.col("gross_booking_value").cast("decimal(12,2)").alias("gross_booking_value"),
        F.col("net_revenue").cast("decimal(12,2)").alias("net_revenue"),
        F.to_timestamp(F.col("created_at")).alias("created_at"),
        F.to_timestamp(F.col("updated_at")).alias("updated_at"),
    )
)


# -------------------------------------------------------------------
# Write Silver Delta table
# -------------------------------------------------------------------

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Created Silver table: {TARGET_TABLE}")
print("Silver row count:", spark.table(TARGET_TABLE).count())

display(spark.table(TARGET_TABLE).limit(10))

In [0]:
TARGET_TABLE = "voyage.silver.fact_marketing_booking"


# -------------------------------------------------------------------
# Validation checks
# -------------------------------------------------------------------

# 1. booking_id duplicates (should be 0)
display(
    spark.sql(f"""
        SELECT COUNT(*) AS booking_id_duplicates
        FROM (
            SELECT booking_id, COUNT(*) AS cnt
            FROM {TARGET_TABLE}
            GROUP BY booking_id
            HAVING COUNT(*) > 1
        )
    """)
)

# 2. orphan tenant_id (should be 0)
display(
    spark.sql(f"""
        SELECT COUNT(*) AS orphan_tenant_id
        FROM {TARGET_TABLE} b
        LEFT JOIN voyage.silver.dim_tenant t
          ON b.tenant_id = t.tenant_id
        WHERE t.tenant_id IS NULL
    """)
)

# 3. orphan property_id (should be 0)
display(
    spark.sql(f"""
        SELECT COUNT(*) AS orphan_property_id
        FROM {TARGET_TABLE} b
        LEFT JOIN voyage.silver.dim_property p
          ON b.property_id = p.property_id
        WHERE p.property_id IS NULL
    """)
)

# 4. orphan conversion_id (should be 0)
display(
    spark.sql(f"""
        SELECT COUNT(*) AS orphan_conversion_id
        FROM {TARGET_TABLE} b
        LEFT JOIN voyage.bronze.website_events e
          ON b.conversion_id = e.event_id
        WHERE e.event_id IS NULL
    """)
)

# 5. orphan session_id (should be 0)
display(
    spark.sql(f"""
        SELECT COUNT(*) AS orphan_session_id
        FROM {TARGET_TABLE} b
        LEFT JOIN voyage.bronze.website_sessions s
          ON b.session_id = s.session_id
        WHERE s.session_id IS NULL
    """)
)

# 6. negative revenue (should be 0)
display(
    spark.sql(f"""
        SELECT COUNT(*) AS negative_revenue
        FROM {TARGET_TABLE}
        WHERE net_revenue < 0
    """)
)

# 7. net_revenue > gross_booking_value (should be 0)
display(
    spark.sql(f"""
        SELECT COUNT(*) AS net_exceeds_gross
        FROM {TARGET_TABLE}
        WHERE net_revenue > gross_booking_value
    """)
)

In [0]:
display(spark.sql(
    f"""SELECT *FROM voyage.dbt_dev.int_booking_touchpoints limit 10;""")
)

In [0]:
display(
    spark.sql(f"""SELECT * FROM voyage.dbt_dev.fct_marketing_attribution""")
)

In [0]:
%sql
select
    booking_id,
    attribution_model,
    round(sum(attributed_net_revenue), 2) as attributed_revenue,
    max(round(net_revenue, 2)) as source_revenue,
    round(
        sum(attributed_net_revenue) - max(net_revenue),
        2
    ) as difference
from voyage.dbt_dev.fct_marketing_attribution
group by
    booking_id,
    attribution_model
having abs(
    round(
        sum(attributed_net_revenue) - max(net_revenue),
        2
    )
) > 0.01
order by abs(
    round(
        sum(attributed_net_revenue) - max(net_revenue),
        2
    )
) desc;